# IFEval on ScreamingFace: stable first run, then the research experiment

IFEval ([arXiv:2311.07911](https://arxiv.org/abs/2311.07911)) is 541 prompts with
machine-checkable constraints — word counts, forbidden punctuation, required sections.
The Engine grades every response with
a deterministic verifier: **no judge model in the grading path, zero grading cost**.

**Mental model: an exam with a mechanical grader.** Every prompt is one exam question
("write 300+ words, no commas, 3 highlighted sections"), and the grader is a script
that counts words and commas — it cannot be argued with and costs nothing. The three
Benchmarks below are three exam FORMATS over the same 541 questions. Running example
for all three: the question is *"describe a cat in exactly two sentences, no commas."*

- `ifeval` — A solo Model writes one answer and tries to follow instructions and
  hands it in. A Fusion is different: its members each write a draft, then the
  synthesizer **blends** the drafts into one NEW answer — and only the blend is
  graded
- `ifeval/self-corrective` — The solo model answers, the grader lists what failed
  ("3 sentences, and there is a comma"), the model then writes its own study note
  ("use exactly two sentences,
  drop the comma") and answers again — up to three attempts, earliest pass wins.
- `ifeval/lanl-ensemble` — (Skurikhin et al., https://openreview.net/forum?id=XSIYfTm2h7).
  Every member's draft is graded INDIVIDUALLY — no blending, ever.
  If member A's draft passes, the case stops
  right there and A's text is submitted **word-for-word**. The synthesizer acts as
  JUDGE, and only in two narrow moments: when TWO OR MORE drafts pass it picks the
  best-written one, and when NOBODY passes it turns the grader's violations into
  coaching text ("A: drop your comma; B: cut to two sentences") for the next of at most
  three rounds. The judge never writes answer text on this exam — so it cannot
  break a constraint a member already satisfied.

One rule to remember: 
- **the synthesizer plays two roles.** Answers blending on `ifeval`
(writes new text; can break constraints)
- judge on `ifeval/lanl-ensemble` (only picks or coaches; cannot).

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

After pulling or merging SDK code, **restart this notebook's kernel before Run
All**. Python keeps already-imported SDK modules in memory; a stale kernel can ask the new Engine
for a pre-merge Benchmark id and receive `unknown_benchmark`.

In [1]:
import screamingface as sf

sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## Stable smoke Candidates

These four cells are a **paid one-Case validation**, not a scientific result. Haiku is the
solo Candidate. The Fusion pairs Haiku with Gemini Flash and uses Flash as its synthesizer,
so the synthesizer is also a direct member — the shape used by Skurikhin et al. ([Ens-1]).

`progress=True` shows the live Engine stream. Raw URL4 node names are expected until
semantic Case/attempt events land.

In [2]:
# Researcher-editable prompt for solo models
SOLO_ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)

# Synthesizer prompt - used when synthesizer writes text (the answers blender on `ifeval`).
# Will be ignored in lanl-ensemble
FUSION_SYNTHESIS_PROMPT = (
    "Produce the final answer to the original request. "
    "Synthesize the strongest supported answer from the panel responses, and follow every "
    "instruction and formatting constraint in the original request."
)

haiku_4_5 = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    prompt=SOLO_ANSWER_PROMPT,
    params={"max_tokens": 4096},
)

kimi_k3 = sf.Model(
    "openrouter/moonshotai/kimi-k3",
    prompt=SOLO_ANSWER_PROMPT,
    params={"max_tokens": 4096},
)

gemini_3_flash = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=SOLO_ANSWER_PROMPT,
    params={"max_tokens": 4096},
)

haiku_gemini_kimi = sf.Fusion(
    members=[haiku_4_5, gemini_3_flash],
    name="haiku-gemini-kimi",
    synthesizer="openrouter/moonshotai/kimi-k3",
    prompt=FUSION_SYNTHESIS_PROMPT,
    params={"max_tokens": 4096},
)
haiku_gemini_kimi

Fusion(['claude-haiku-4.5', 'gemini-3-flash-preview'], name='haiku-gemini-kimi', synthesizer='openrouter/moonshotai/kimi-k3', prompt='Produce the final answer to the original request. Synthesize the strongest supported answer from the panel responses, and follow every instruction and formatting constraint in the original request.', params={'max_tokens': 4096})

## ① Baseline — one model, one shot

In [ ]:
ifeval = sf.evaluate(
    kimi_k3,
    benchmark="ifeval",
    limit=1,  # only eval on 1 row
    progress=True,
)
ifeval.to_dict()

## ② Does blending preserve instructions?

In [ ]:
ifeval_fusion = sf.evaluate(
    haiku_gemini_kimi,
    benchmark="ifeval",
    limit=1,
    progress=True,
)
ifeval_fusion.to_dict()

## ③ Can a model correct itself?

Will need more output tokens

In [4]:
kimi_k3_more_tokens = sf.Model(
    "openrouter/moonshotai/kimi-k3",
    prompt=SOLO_ANSWER_PROMPT,
    params={"max_tokens": 16384},  # reasoning headroom for corrective attempts
)

ifeval_self_corrective = sf.evaluate(
    kimi_k3_more_tokens,
    benchmark="ifeval/self-corrective",
    limit=10,
    progress=True,
)
ifeval_self_corrective.to_dict()

ScreamingFace · Run started
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 28s · 202 in / 896 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 50s · 202 in / 1,660 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 56s · 159 in / 1,755 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 70s · 137 in / 585 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 76s · 201 in / 2,467 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 83s · 201 in / 1,959 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 92s · 179 in / 586 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 95s · 179 in / 433 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 99s · 180 in / 1,519 out · stop
ScreamingFace · Model completed · openrouter/moonshotai/kimi-k3 · 103s · 180 in / 771 out · stop
S

{'schema': 'screamingface.report.v1',
 'started_at': '2026-08-10T08:08:59.125750Z',
 'completed_at': '2026-08-10T08:11:06.722195Z',
 'benchmark': {'id': 'ifeval/self-corrective',
  'revision': '6d017e4193b84f70',
  'case_count': 10},
 'candidates': [{'run_id': 'MD3u8swa0BeWEn4hZXlPflKyUVqkwTqXONuYCMBVCoFovG2r3r9CR9UdjsirRxqj',
   'started_at': '2026-08-10T08:08:59.125750Z',
   'completed_at': '2026-08-10T08:11:06.722195Z',
   'name': 'kimi-k3',
   'kind': 'model',
   'url4': "(candidate:0.0:'(model_1:0.0:/openrouter/moonshotai/kimi-k3?max_tokens=16384&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\')!\\'$model_1\\'', (rows:0.0:(selected_rows:0.0:/benchmarks/ifeval/76b73fe25ac36ae1/cases*(checked:0.0:(answer_1:0.0:(candidate_result:0.0:/candidate?web_search=false&q=($item.input)!'$candidate')!'$candidate_result', check_1:0.0:/benchmarks/ifeval/76b73fe25ac36ae1/check($answer_1)!'$item.id:1', feedback_1:0.0:/

## ④ The verifying ensemble (the paper's protocol)

Members answer, the checker checks **each draft individually**, and the synthesizer —
acting as judge here — picks a passing answer verbatim, or coaches everyone and retries
when nobody passed. A judge (synthesizer model) never rewrites the output text

In [3]:
ifeval_lanl_fusion = sf.evaluate(
    haiku_gemini_kimi,
    benchmark="ifeval/lanl-ensemble",
    limit=10,
    progress=True,
)
ifeval_lanl_fusion.to_dict()

ScreamingFace · Run started
ScreamingFace · Model completed · openrouter/google/gemini-3-flash-preview · 2.0s · 42 in / 28 out · stop
ScreamingFace · Model completed · openrouter/anthropic/claude-haiku-4.5 · 2.4s · 51 in / 60 out · stop
ScreamingFace · Model completed · openrouter/google/gemini-3-flash-preview · 3.7s · 63 in / 243 out · stop
ScreamingFace · Model completed · openrouter/anthropic/claude-haiku-4.5 · 4.5s · 75 in / 313 out · stop
ScreamingFace · Model completed · openrouter/google/gemini-3-flash-preview · 5.4s · 104 in / 146 out · stop
ScreamingFace · Model completed · openrouter/anthropic/claude-haiku-4.5 · 6.1s · 117 in / 332 out · stop
ScreamingFace · Model completed · openrouter/anthropic/claude-haiku-4.5 · 9.2s · 103 in / 399 out · stop
ScreamingFace · Model completed · openrouter/google/gemini-3-flash-preview · 10s · 97 in / 479 out · stop
ScreamingFace · Model completed · openrouter/anthropic/claude-haiku-4.5 · 11s · 67 in / 334 out · stop
ScreamingFace · Model com

{'schema': 'screamingface.report.v1',
 'started_at': '2026-08-10T07:37:17.260480Z',
 'completed_at': '2026-08-10T07:39:21.874396Z',
 'benchmark': {'id': 'ifeval/lanl-ensemble',
  'revision': '7ffaa303274c970d',
  'case_count': 10},
 'candidates': [{'run_id': 'xgzSwYXOjNcjHjCk5cMfjteXZtuh7rOO8LUYPVai78UOZvMymca43F3Itr1Rh6X6',
   'started_at': '2026-08-10T07:37:17.260480Z',
   'completed_at': '2026-08-10T07:39:21.874396Z',
   'name': 'haiku-gemini-kimi',
   'kind': 'fusion',
   'url4': "(candidate_synthesizer:0.0:'(model_1:0.0:/openrouter/moonshotai/kimi-k3?max_tokens=4096&web_search=false&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\')!\\'$model_1\\'', candidate_member_1:0.0:'(model_1:0.0:/openrouter/anthropic/claude-haiku-4.5?max_tokens=4096&q=($input)!\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\')!\\'$model_1\\'', candidate_member_